In [ ]:
# TODO: create a separate module for helper functions

## Introduction

In this notebook you'll be working with behavioral data collected from two experiments in which two mice foraged for food in an environment with three patches whose reward rates changed dynamically over time.

The experiments consist of three phases each:

1. A "presocial" phase, in which each mouse was in the environment alone for 3-4 days.
2. A "social" phase, in which both mice were in the environment together for 2 weeks.
3. A "postsocial" phase, in which each mouse was in the environment alone again for 3-4 days.

The goal of the experiment was to understand how mouse behavior changes as they learn to forage for food in the environment, and how their behavior differs in social vs. solo settings.

## Set up

### Prerequisities

1. [VSCode](https://code.visualstudio.com/download) (re: forks, e.g. Cursor or Windsurf: Cursor does support remote tunneling and is fine to use, Windsurf and most others will NOT WORK as they don't support remote tunneling)

  - Ensure you have the 'Python' and 'Jupyter' VSCode extensions installed.

2. HPC account credentials:

    - tricentre0
        - tgKOfI*741

    - tricentre1
        - l2b£l6S$98

    - tricentre2
        - p[p9K1b(05
    
    - tricentre3
        - g9-21S&q69

    - tricentre4
        - 5Zh.5a0Js=
    
    - tricentre5
        - NM5@x24<i3
    
    - tricentre6
        - 90+£9Mh21<
    
    - tricentre7
        - m3Z}$wT165
    
    - tricentre8
        - 7ez04]Q7m4
    
    - tricentre9
        - 3Nli;F6Ri9


### Connection Instructions

1. Within VSCode, ssh into hpc-gw2: 

    ```bash
    ssh -J <tricentre_account>@ssh.swc.ucl.ac.uk <tricentre_account>@hpc-gw2
    ``` 
    (you will be prompted to enter the account's password)

2. In the VSCode terminal, start a remote tunnel on a hpc compute node
    ```bash
    srun --nbuffered --ntasks 8 --mem 32G --pty /usr/bin/bash -i  # connect to compute node
    code tunnel # start tunnel on compute node
    ```
    and follow the instructions to open a new VSCode Window in the remote tunnel.

3. In the VSCode file explorer, open `/ceph/aeon/aeon/tricentre_hackathon` (the base directory for this project)

4. Open this notebook (`tricentre_hackathon.ipynb`), select the 'aeon' environment in the kernel picker, and try running the first code cell (with all the `import`s)

<img src="./tricentre_hackathon_assets/example_foraging_over_blocks.png" width="100%">

<img src="./tricentre_hackathon_assets/social02_env_protocol.png" width="100%">

In [ ]:
from IPython.display import HTML

# Construct an iframe that uses srcdoc to embed Mermaid, escaping single quotes in the diagram init config
html = """
<iframe
  srcdoc='<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8">
  <script src="https://cdn.jsdelivr.net/npm/mermaid/dist/mermaid.min.js"></script>
  <script>mermaid.initialize({ startOnLoad: true, theme: "dark" });</script>
</head>
<body>
  <div class="mermaid">
    %%{init: {&#39;theme&#39;: &#39;dark&#39;}}%%
    gantt
      title Social0.2
      dateFormat  YYYY-MM-DD

      section Aeon3
      BAA-1104045             :2024-01-31, 2024-02-03
      Clean                   :2024-02-04, 2024-02-05
      BAA-1104047             :2024-02-05, 2024-02-08
      Clean                   :2024-02-08, 2024-02-09
      Tube Test               :2024-02-09, 2024-02-10
      BAA-1104045 + BAA-1104047:2024-02-09, 2024-02-23
      Clean                   :2024-02-23, 2024-02-24
      BAA-1104045             :2024-02-25, 2024-02-28
      Clean                   :2024-02-28, 2024-02-29
      BAA-1104047             :2024-02-28, 2024-03-02

      section Aeon4
      BAA-1104048             :2024-01-31, 2024-02-03
      Clean                   :2024-02-04, 2024-02-05
      BAA-1104049             :2024-02-05, 2024-02-08
      Clean                   :2024-02-08, 2024-02-09
      Tube Test               :2024-02-09, 2024-02-10
      BAA-1104048 + BAA-1104049:2024-02-09, 2024-02-23
      Clean                   :2024-02-23, 2024-02-24
      BAA-1104048             :2024-02-25, 2024-02-28
      Clean                   :2024-02-28, 2024-02-29
      BAA-1104049             :2024-02-28, 2024-03-02
  </div>
</body>
</html>'
  style="width:100%; height:600px; border:0;"
></iframe>
"""

display(HTML(html))


In [ ]:
%load_ext autoreload
%autoreload 2
# %flow mode reactive

# TODO: there will probably be a number of unecessary imports here, clean it up later

import datetime
import sys
import os
import warnings
from IPython.display import display, Markdown
from pathlib import Path
from typing import Any, Tuple, List, Dict

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objs as go
import statsmodels.api as sm
from plotly.subplots import make_subplots
from tqdm.notebook import tqdm

import datajoint as dj
dj.config['database.host'] = "aeon-db2"
dj.config['database.user'] = "aeon-tri2025"
dj.config['database.password'] = "hackathon-tri2025"
from aeon.dj_pipeline.analysis.block_analysis import *
from aeon.dj_pipeline import acquisition, streams, subject
from swc.aeon.io import api as aeon_api
from aeon.schema.schemas import social02

cwd = os.getcwd()
project_path = os.path.join(cwd, "ProjectAeon", "aeon_scratchpad", "aeon_analysis", "aeon_methods_paper")
sys.path.append(project_path)
from data_io_utils_hackathon import load_all_patch_data, load_all_foraging_bouts, load_all_position_data

TODO: finish this

## Data Overview & Example Analyses

You'll be working with data that contains: 

1. The full-pose position and ID labels of each mouse in the environment.
2. Details on the activity of each mouse in a patch, per block (e.g. time spent in patch, distance foraged in patch, food pellets accumulated in patch, and running and overall patch preference based on these metrics).
3. Details on the patch properties, per block (e.g. patch location, patch reward rate).

For simplicitly, excluded from this example dataset are the continuous raw video (9 cameras), audio (2 microphones), rfid data (9 readers), and weight data (weight scale in the nest) recorded during the experiments.

Some example analyses are:

...


Below you'll see some starter examples to load and visualize the data.

## Example Analysis

In [ ]:
cm2px = 5.2  # 1 cm = 5.2 px roughly in aeon arenas
light_off, light_on = 7, 20  # i.e., light is off from 7am to 8pm
# TODO: @JAI should we also give them the arena details here? Eg arena radius, arena center, patch coordinates, nest etc?

In [ ]:
# Uncomment to load all the data
# experiments = [
#     {"name": "social0.2-aeon3", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 16:00:00', "social_end": '2024-02-23 13:00:00', "postsocial_start": '2024-02-25 17:00:00', "postsocial_end": '2024-03-02 14:00:00'},
#     {"name": "social0.2-aeon4", "presocial_start": '2024-01-31 11:00:00', "presocial_end": '2024-02-08 15:00:00', "social_start": '2024-02-09 17:00:00', "social_end": '2024-02-23 12:00:00', "postsocial_start": '2024-02-25 18:00:00', "postsocial_end": '2024-03-02 13:00:00'},
# ]

In [ ]:
# For testing purposes, every period 1 day long
# TODO: maybe delete now that position data loading is rather fast?
experiments = [
    {"name": "social0.2-aeon3", "presocial_start": '2024-02-01 07:00:00', "presocial_end": '2024-02-02 07:00:00', "social_start": '2024-02-10 07:00:00', "social_end": '2024-02-11 07:00:00', "postsocial_start": '2024-02-27 07:00:00', "postsocial_end": '2024-02-28 07:00:00'},
    {"name": "social0.2-aeon4", "presocial_start": '2024-02-01 07:00:00', "presocial_end": '2024-02-02 07:00:00', "social_start": '2024-02-10 07:00:00', "social_end": '2024-02-11 07:00:00', "postsocial_start": '2024-02-27 07:00:00', "postsocial_end": '2024-02-28 07:00:00'},
]

## Patch data

In [ ]:
patch_info_dict, subject_patch_data_dict, subject_patch_pref_dict = load_all_patch_data(experiments)

In [ ]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract example DataFrames
df1 = patch_info_dict[exp_name][period]
df2 = subject_patch_data_dict[exp_name][period]
df3 = subject_patch_pref_dict[exp_name][period]

# Display the first few rows of each
md = (
    f"### Patch Info — {exp_name}, {period}\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `wheel_timestamps`: (@JAI not sure what this is ngl)\n"
    "- `patch_rate`: Rate of the patch (@JAI you may want to add more detail but I guess the rate and oddset are explained in the slide. Maybe we should give them the easy/medium/hard patch rates? Though they don't change in social 0.2 do they so it's pretty straightforward?) \n"
    "- `patch_offset`: Offset of the patch\n"
)
display(Markdown(md))
display(df1.head())

md = (
    f"### Subject Patch Data — {exp_name}, {period}\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `subject_name`: Name of the subject\n"
    "- `in_patch_timestamps`: Array of timestamps when the subject was in the patch (@JAI do we want to define this/add a radius that was used/specfy sampling rate?)\n"
    "- `in_patch_time`: Total time spent in the patch (in seconds) (@JAI not sure of the unit here)\n"
    "- `in_patch_rfid_timestamps`: Array of timestamps when the subject was detected in the patch via RFID\n"
    "- `pellet_counts`: Number of pellets delivered\n"
    "- `pellet_timestamps`: Array of timestamps when pellets were delivered\n"
    "- `patch_threshold`: Array of thresholds at which each pellet was delivered (@JAI not sure I explained this well)\n"
    "- `wheel_cumsum_distance_travelled`: Cumulative distance that the foraging patch wheel was spun (in cm)\n"
)
display(Markdown(md))
display(df2.head())

md = (
    f"### Subject Patch Preference — {exp_name}, {period}\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n"
    "- `block_start`: Start time of the block\n"
    "- `patch_name`: Name of the patch\n"
    "- `subject_name`: Name of the subject\n"
    "- `cumulative_preference_by_time`: (@JAI can you fill these in?)\n"
    "- `cumulative_preference_by_wheel`:\n"
    "- `running_preference_by_time`:\n"
    "- `running_preference_by_wheel`:\n"
    "- `final_preference_by_time`:\n"
    "- `final_preference_by_wheel`:\n"
)
display(Markdown(md))
display(df3.head())

In [ ]:
# TODO: add example plots/uses for the patch_info and subject_patch_pref DataFrames

In [ ]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the subject patch data
df = subject_patch_data_dict[exp_name][period]

# Ensure data is sorted and ready
dt_seconds = 0.02
subjects = sorted(df['subject_name'].unique())
patches = sorted(df['patch_name'].unique())

fig = go.Figure()

# Build each trace (continuous + Δ>0.5 downsample)
for (subject, patch), grp in df.groupby(['subject_name', 'patch_name']):
    grp = grp.sort_values('block_start')
    total_n = sum(len(a) for a in grp.wheel_cumsum_distance_travelled)

    times = np.empty(total_n, dtype='datetime64[ns]')
    dists = np.empty(total_n, dtype=float)
    idx, offset = 0, 0.0

    for bs_val, arr in zip(grp.block_start, grp.wheel_cumsum_distance_travelled):
        arr = np.asarray(arr)
        n = arr.size
        offs = (np.arange(n) * dt_seconds * 1e9).astype('timedelta64[ns]')
        times[idx:idx+n] = np.datetime64(bs_val) + offs
        dists[idx:idx+n] = arr + offset
        offset += arr[-1]
        idx += n

    # Downsample by change > 0.5
    diffs = np.abs(np.diff(dists, prepend=dists[0]))
    mask = diffs > 0.5
    mask[0] = True

    fig.add_trace(
        go.Scatter(
            x=times[mask],
            y=dists[mask] / 100,  # convert to meters
            mode='lines',
            name=f"{subject} — {patch}",
            line=dict(width=1.5)
        )
    )

# Styling (match px.timeline aesthetic)
fig.update_xaxes(
    showgrid=False,
    zeroline=False,
    showline=True,
    ticks='outside',
    showticklabels=True
)

fig.update_yaxes(
    title_text="Distance spun on wheel (m)",
    showgrid=False,
    zeroline=False,
    showline=True,
    ticks='outside',
    showticklabels=True
)

fig.update_layout(
    template="simple_white",
    plot_bgcolor="white",
    showlegend=True
)

fig.show(config={'staticPlot': True})

In [ ]:
foraging_data_dict = load_all_foraging_bouts(experiments)

In [ ]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the foraging DataFrame
df = foraging_data_dict[exp_name][period]

# Display the first few rows
md = (
    f"### Foraging Data — {exp_name}, {period}\n"
    "Columns:\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `start`: Start time of the foraging bout\n"
    "- `end`: End time of the foraging bout\n"
    "- `n_pellets`: Number of pellets consumed during the bout\n"
    "- `cum_wheel_dist`: Cumulative distance spun on the wheel during the bout (in cm)\n"
    "- `subject`: Subject name\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n\n"
)
display(Markdown(md))
display(df.head())

In [ ]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Get the foraging data
df = foraging_data_dict[exp_name][period]

# Ensure correct types
df['start'] = pd.to_datetime(df['start'])
df['end'] = pd.to_datetime(df['end'])

# Plot
subjects = sorted(df['subject'].unique())

fig = px.timeline(
    df,
    x_start="start",
    x_end="end",
    y="subject",
    hover_data=["n_pellets", "cum_wheel_dist"],
    category_orders={"subject": subjects}
)

# Styling
fig.update_traces(
    opacity=1,
    marker_color="#555555",
    marker_line_color="#555555",
    marker_line_width=1.5
)

fig.update_layout(
    template='simple_white',
    plot_bgcolor='white',
    margin=dict(l=150, r=20, t=20, b=20),
    height=max(100, len(subjects) * 25 + 50),
    xaxis=dict(
        showgrid=False, zeroline=False,
        showline=True, ticks='outside', showticklabels=False
    ),
    yaxis=dict(
        showgrid=False, zeroline=False,
        showline=False, ticks='', showticklabels=True,
        title=''
    )
)

fig.show()

## Position data

In [ ]:
position_data = load_all_position_data(experiments) # TODO: load the denoised data

In [ ]:
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Extract the position DataFrame
df = position_data_dict[exp_name][period]

# Display the first few rows
md = (
    f"### Position Data — {exp_name}, {period}\n"
    "Columns:\n"
    "- `time (index)`: Timestamp of the position data\n"
    "- `experiment_name`: Name of the experiment\n"
    "- `identity_name`: Name of the tracked identity (e.g., mouse)\n"
    "- `identity_likelihood`: Likelihood score for the tracked identity\n"
    "- `x`: X coordinate of the centroid position (in pixels)\n"
    "- `y`: Y coordinate of the centroid position (in pixels)\n"
    "- `likelihood`: Likelihood score for the (x,y) position\n"
    "- `anchor_part`: Anchor part used for tracking (centroid of the mouse)\n"
    "- `period`: Period of the experiment (e.g., presocial, social, postsocial)\n\n"
)
display(Markdown(md))
display(df.head())

In [ ]:
# TODO: Fix this so it works
exp_name = experiments[0]["name"]  # Choose the experiment to visualize
period = "social"  # Choose the period to visualise ("presocial", "social", or "postsocial")

# Load and clean the data
df = position_data_dict[exp_name][period]
df = df.dropna(subset=['x', 'y']).reset_index()

# Add a 'date' column for grouping by day
timestamp_col = df.columns[0]
df['date'] = pd.to_datetime(df[timestamp_col]).dt.date

# Define bin size and grid
bin_size = 25  # pixels
x_bins = np.arange(df['x'].min(), df['x'].max() + bin_size, bin_size)
y_bins = np.arange(df['y'].min(), df['y'].max() + bin_size, bin_size)

# Plot a heatmap for each day
for day, day_df in df.groupby('date'):
    heatmap, x_edges, y_edges = np.histogram2d(
        day_df['x'],
        day_df['y'],
        bins=[x_bins, y_bins]
    )

    fig = go.Figure(data=go.Heatmap(
        z=heatmap.T,  # transpose to align axes correctly
        x=x_edges[:-1],
        y=y_edges[:-1],
        colorscale='Greys',
        colorbar=dict(title='Count'),
        showscale=True
    ))

    fig.update_layout(
        title=f"Position Heatmap — {exp_name}, {period} ({day})",
        xaxis=dict(
            title='x (pixels)',
            showgrid=False,
            showticklabels=False,
            zeroline=False
        ),
        yaxis=dict(
            title='y (pixels)',
            showgrid=False,
            showticklabels=False,
            zeroline=False,
            scaleanchor='x'
        ),
        template='simple_white',
        plot_bgcolor='white',
        margin=dict(l=40, r=40, t=40, b=40),
        width=500,
        height=500
    )

    fig.show()